In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║     SVD CONFLUENCE — ML MODEL v4.0 (TRAILING SL + AI METRICS)   ║
║   Input : ketqua.csv  (Dữ liệu điểm hợp lưu)                     ║
║          data.csv    (OHLCV gốc để backtest)                     ║
║   Output: predictions_final.csv + trading_report.html            ║
╚══════════════════════════════════════════════════════════════════╝

Nâng cấp chính so với v3.2:
- Thêm kéo SL / trailing stop cho cả LONG và SHORT.
- Tách riêng hàm simulate_trade() để không lặp code backtest nhiều lần.
- Giữ nguyên giao diện bảng gốc: STT, Tập dữ liệu, Thời gian, Entry,
  Tín hiệu, Độ tin cậy, SL, TP, Kết quả, PnL.
- Cột SL mặc định hiển thị SL ban đầu như ảnh gốc; trailing SL được dùng
  trong tính toán kết quả/PnL. Nếu muốn hiện SL sau khi kéo, bật
  SHOW_TRAILED_SL_IN_SL_COLUMN = True.
"""

from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    USE_XGB = True
except ImportError:
    USE_XGB = False

# ════════════════════════════════════════════════════════════════
# 0. CẤU HÌNH CHIẾN THUẬT
# ════════════════════════════════════════════════════════════════
DATA_FILE = "data.csv"
SIGNAL_FILE = "ketqua.csv"
OUTPUT_CSV = "predictions_final.csv"
OUTPUT_HTML = "trading_report.html"

SL_POINTS = 7.0
TP_POINTS = 14.0
MAX_CANDLES_FWD = 100
TRAIN_RATIO = 0.70

# AI: nếu xác suất thắng theo hướng scanner >= ngưỡng thì giữ hướng scanner,
# nếu thấp hơn ngưỡng thì đảo chiều.
AI_PROBA_THRESHOLD = 0.50

# Kéo SL / trailing stop
USE_TRAILING_SL = True
MOVE_SL_TO_BREAKEVEN = True
BREAKEVEN_TRIGGER_POINTS = 7.0     # lời >= 7 điểm thì kéo SL về hòa vốn
BREAKEVEN_OFFSET_POINTS = 0.0      # 0 = đúng entry; 0.5 = entry +0.5 cho LONG / entry -0.5 cho SHORT
TRAIL_START_POINTS = 7.0           # bắt đầu trailing khi lời >= 7 điểm
TRAIL_DISTANCE_POINTS = 7.0        # SL cách đỉnh/đáy thuận chiều 7 điểm
TRAIL_STEP_POINTS = 1.0            # chỉ kéo SL khi cải thiện tối thiểu 1 điểm
SHOW_TRAILED_SL_IN_SL_COLUMN = False

# Vì OHLC không biết trong 1 nến giá chạm High trước hay Low trước,
# backtest giữ logic giống code cũ: ưu tiên TP trước SL trong cùng một cây nến.
# Trailing SL được cập nhật sau khi cây nến hiện tại đóng để giảm ảo tưởng intrabar.

print("=" * 70)
print("  SVD CONFLUENCE — ML MODEL v4.0 (Trailing SL + AI Metrics)")
print("=" * 70)


# ════════════════════════════════════════════════════════════════
# 1. HÀM TIỆN ÍCH
# ════════════════════════════════════════════════════════════════
def require_file(path: str) -> None:
    if not Path(path).exists():
        raise FileNotFoundError(f"Không tìm thấy file '{path}'!")


def normalize_direction(value) -> str:
    text = str(value).upper()
    if "LONG" in text or "UP" in text or "BUY" in text:
        return "LONG"
    return "SHORT"


def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)


def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "volume" not in df.columns:
        df["volume"] = 1.0

    df["sma_20"] = df["close"].rolling(20).mean()
    df["ema_20"] = df["close"].ewm(span=20, adjust=False).mean()
    df["return_1"] = df["close"].pct_change(1).replace([np.inf, -np.inf], np.nan).fillna(0)
    df["return_3"] = df["close"].pct_change(3).replace([np.inf, -np.inf], np.nan).fillna(0)
    df["vol_avg_10_prev"] = df["volume"].shift(1).rolling(10).mean()

    prev_close = df["close"].shift(1)
    tr_1 = df["high"] - df["low"]
    tr_2 = (df["high"] - prev_close).abs()
    tr_3 = (df["low"] - prev_close).abs()
    true_range = pd.concat([tr_1, tr_2, tr_3], axis=1).max(axis=1)
    df["atr_14"] = true_range.rolling(14).mean()

    if "rsi_14" not in df.columns:
        df["rsi_14"] = compute_rsi(df["close"], 14)

    numeric_cols = ["sma_20", "ema_20", "vol_avg_10_prev", "atr_14", "rsi_14"]
    for col in numeric_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].ffill().bfill()

    return df


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    require_file(SIGNAL_FILE)
    require_file(DATA_FILE)

    kq = pd.read_csv(SIGNAL_FILE, encoding="utf-8-sig")
    kq.columns = [c.strip() for c in kq.columns]

    column_mapping = {
        "Nến thứ": "candle_idx",
        "Thời gian": "time",
        "Hướng lệnh": "direction",
        "Giá Open": "open",
        "Giá High": "high",
        "Giá Low": "low",
        "Giá Close": "close",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
        "RSI 14": "rsi_14",
    }
    kq = kq.rename(columns=column_mapping)

    if "candle_idx" not in kq.columns and len(kq.columns) > 0:
        kq.rename(columns={kq.columns[0]: "candle_idx"}, inplace=True)

    df = pd.read_csv(DATA_FILE)
    df.columns = [c.lower().strip() for c in df.columns]

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
    elif "date" in df.columns:
        df["time"] = pd.to_datetime(df["date"], errors="coerce")
    else:
        df["time"] = pd.RangeIndex(start=0, stop=len(df), step=1)

    required_ohlc = ["open", "high", "low", "close"]
    missing = [col for col in required_ohlc if col not in df.columns]
    if missing:
        raise ValueError(f"data.csv đang thiếu cột bắt buộc: {missing}")

    df = df.sort_values("time").reset_index(drop=True)
    df = add_indicators(df)

    kq["candle_idx"] = pd.to_numeric(kq["candle_idx"], errors="coerce")
    kq = kq.dropna(subset=["candle_idx"]).copy()
    kq["candle_idx"] = kq["candle_idx"].astype(int)
    kq = kq.sort_values("candle_idx").reset_index(drop=True)

    safe_idx = kq["candle_idx"].clip(0, len(df) - 1).astype(int).to_numpy()

    # Bổ sung dữ liệu từ data.csv nếu ketqua.csv chưa có hoặc bị rỗng.
    cols_to_pull = [
        "time", "open", "high", "low", "close", "volume", "rsi_14",
        "sma_20", "ema_20", "atr_14", "return_1", "return_3", "vol_avg_10_prev"
    ]
    for col in cols_to_pull:
        source_values = df[col].iloc[safe_idx].to_numpy()
        if col not in kq.columns:
            kq[col] = source_values
        else:
            kq[col] = kq[col].where(~pd.isna(kq[col]), source_values)

    if "direction" not in kq.columns:
        raise ValueError("ketqua.csv đang thiếu cột hướng lệnh: 'Hướng lệnh' hoặc 'direction'.")

    numeric_cols = [
        "open", "high", "low", "close", "volume", "rsi_14",
        "sma_20", "ema_20", "atr_14", "return_1", "return_3", "vol_avg_10_prev"
    ]
    for col in numeric_cols:
        kq[col] = pd.to_numeric(kq[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        kq[col] = kq[col].ffill().bfill()

    return kq, df


kq, df = load_data()

high_all = df["high"].astype(float).to_numpy()
low_all = df["low"].astype(float).to_numpy()
close_all = df["close"].astype(float).to_numpy()
open_all = df["open"].astype(float).to_numpy()


# ════════════════════════════════════════════════════════════════
# 2. BACKTEST CORE: CÓ KÉO SL
# ════════════════════════════════════════════════════════════════
def classify_result(pnl: float, exit_reason: str) -> str:
    if exit_reason == "OPEN":
        return "OPEN"
    if pnl > 0:
        return "WIN"
    if pnl < 0:
        return "LOSS"
    return "BE"


def simulate_trade(candle_idx: int, signal: str, use_trailing: bool = True) -> dict:
    """
    Mô phỏng 1 lệnh theo OHLC.
    - TP cố định theo TP_POINTS.
    - SL ban đầu theo SL_POINTS.
    - Nếu use_trailing=True, active_sl sẽ được kéo theo hướng có lợi.
    """
    if candle_idx >= len(close_all) - 1:
        return {
            "entry": np.nan,
            "initial_sl": np.nan,
            "final_sl": np.nan,
            "tp": np.nan,
            "exit_price": np.nan,
            "exit_idx": None,
            "exit_reason": "OPEN",
            "outcome": "OPEN",
            "pnl": 0.0,
        }

    signal = normalize_direction(signal)
    is_long = signal == "LONG"
    entry = float(close_all[candle_idx])

    initial_sl = entry - SL_POINTS if is_long else entry + SL_POINTS
    tp = entry + TP_POINTS if is_long else entry - TP_POINTS
    active_sl = initial_sl

    end_idx = min(candle_idx + MAX_CANDLES_FWD + 1, len(close_all))

    for j in range(candle_idx + 1, end_idx):
        high = float(high_all[j])
        low = float(low_all[j])

        if is_long:
            # Giữ logic gốc: nếu cùng nến chạm TP và SL, ưu tiên TP.
            if high >= tp:
                pnl = TP_POINTS
                return {
                    "entry": entry,
                    "initial_sl": round(initial_sl, 2),
                    "final_sl": round(active_sl, 2),
                    "tp": round(tp, 2),
                    "exit_price": round(tp, 2),
                    "exit_idx": j,
                    "exit_reason": "TP",
                    "outcome": "WIN",
                    "pnl": round(pnl, 2),
                }

            if low <= active_sl:
                pnl = active_sl - entry
                exit_reason = "TRAIL_SL" if active_sl > initial_sl else "SL"
                return {
                    "entry": entry,
                    "initial_sl": round(initial_sl, 2),
                    "final_sl": round(active_sl, 2),
                    "tp": round(tp, 2),
                    "exit_price": round(active_sl, 2),
                    "exit_idx": j,
                    "exit_reason": exit_reason,
                    "outcome": classify_result(pnl, exit_reason),
                    "pnl": round(pnl, 2),
                }

            if use_trailing and USE_TRAILING_SL:
                favorable_move = high - entry

                if MOVE_SL_TO_BREAKEVEN and favorable_move >= BREAKEVEN_TRIGGER_POINTS:
                    active_sl = max(active_sl, entry + BREAKEVEN_OFFSET_POINTS)

                if favorable_move >= TRAIL_START_POINTS:
                    candidate_sl = high - TRAIL_DISTANCE_POINTS
                    if candidate_sl - active_sl >= TRAIL_STEP_POINTS:
                        active_sl = max(active_sl, candidate_sl)

        else:
            # SHORT
            if low <= tp:
                pnl = TP_POINTS
                return {
                    "entry": entry,
                    "initial_sl": round(initial_sl, 2),
                    "final_sl": round(active_sl, 2),
                    "tp": round(tp, 2),
                    "exit_price": round(tp, 2),
                    "exit_idx": j,
                    "exit_reason": "TP",
                    "outcome": "WIN",
                    "pnl": round(pnl, 2),
                }

            if high >= active_sl:
                pnl = entry - active_sl
                exit_reason = "TRAIL_SL" if active_sl < initial_sl else "SL"
                return {
                    "entry": entry,
                    "initial_sl": round(initial_sl, 2),
                    "final_sl": round(active_sl, 2),
                    "tp": round(tp, 2),
                    "exit_price": round(active_sl, 2),
                    "exit_idx": j,
                    "exit_reason": exit_reason,
                    "outcome": classify_result(pnl, exit_reason),
                    "pnl": round(pnl, 2),
                }

            if use_trailing and USE_TRAILING_SL:
                favorable_move = entry - low

                if MOVE_SL_TO_BREAKEVEN and favorable_move >= BREAKEVEN_TRIGGER_POINTS:
                    active_sl = min(active_sl, entry - BREAKEVEN_OFFSET_POINTS)

                if favorable_move >= TRAIL_START_POINTS:
                    candidate_sl = low + TRAIL_DISTANCE_POINTS
                    if active_sl - candidate_sl >= TRAIL_STEP_POINTS:
                        active_sl = min(active_sl, candidate_sl)

    return {
        "entry": entry,
        "initial_sl": round(initial_sl, 2),
        "final_sl": round(active_sl, 2),
        "tp": round(tp, 2),
        "exit_price": np.nan,
        "exit_idx": None,
        "exit_reason": "OPEN",
        "outcome": "OPEN",
        "pnl": 0.0,
    }


# ════════════════════════════════════════════════════════════════
# 3. CHIA TẬP TRAIN / PREDICT & GÁN NHÃN
# ════════════════════════════════════════════════════════════════
split = max(3, int(len(kq) * TRAIN_RATIO))
kq_train = kq.iloc[:split].reset_index(drop=True)
kq_pred = kq.iloc[split:].reset_index(drop=True)


def assign_label(candle_idx: int, direction_str) -> int:
    result = simulate_trade(candle_idx, normalize_direction(direction_str), use_trailing=True)
    if result["outcome"] == "OPEN":
        return -1
    return 1 if result["pnl"] > 0 else 0


kq_train["label"] = [
    assign_label(int(row["candle_idx"]), row["direction"])
    for _, row in kq_train.iterrows()
]

kq_train_valid = kq_train[kq_train["label"] != -1].reset_index(drop=True)
if len(kq_train_valid) == 0:
    raise ValueError("Không có lệnh train nào đóng trong MAX_CANDLES_FWD. Hãy tăng MAX_CANDLES_FWD hoặc kiểm tra dữ liệu.")


# ════════════════════════════════════════════════════════════════
# 4. TRÍCH XUẤT TÍNH NĂNG & TRAIN MODEL
# ════════════════════════════════════════════════════════════════
def build_features(row: pd.Series) -> dict:
    candle_range = max(float(row["high"] - row["low"]), 0.5)
    close = max(float(row["close"]), 1.0)
    volume = max(float(row["volume"]), 0.0)
    vol_avg = max(float(row["vol_avg_10_prev"]), 1.0)
    scanner_dir = 1 if normalize_direction(row["direction"]) == "LONG" else 0

    body = float(row["close"] - row["open"])
    upper_wick = float(row["high"] - max(row["open"], row["close"]))
    lower_wick = float(min(row["open"], row["close"]) - row["low"])

    return {
        "scanner_direction": scanner_dir,
        "rsi_14": float(row["rsi_14"]),
        "vol_ratio_log": np.log1p(volume / vol_avg),
        "sma_diff_norm": float(row["close"] - row["sma_20"]) / candle_range,
        "ema_diff_norm": float(row["close"] - row["ema_20"]) / candle_range,
        "body_norm": body / candle_range,
        "upper_wick_norm": upper_wick / candle_range,
        "lower_wick_norm": lower_wick / candle_range,
        "candle_range_pct": candle_range / close,
        "atr_norm": float(row["atr_14"]) / close,
        "return_1": float(row["return_1"]),
        "return_3": float(row["return_3"]),
    }


FEATURE_COLUMNS = list(build_features(kq_train_valid.iloc[0]).keys())
X_train_df = pd.DataFrame([build_features(row) for _, row in kq_train_valid.iterrows()])[FEATURE_COLUMNS]
y_train = kq_train_valid["label"].astype(int).to_numpy()
X_pred_df = pd.DataFrame([build_features(row) for _, row in kq_pred.iterrows()])[FEATURE_COLUMNS]

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_clean = imputer.fit_transform(X_train_df)
X_pred_clean = imputer.transform(X_pred_df)
X_train_s = scaler.fit_transform(X_train_clean)
X_pred_s = scaler.transform(X_pred_clean)

if len(np.unique(y_train)) < 2:
    # Dữ liệu train chỉ có 1 loại nhãn thì không thể học phân loại thật sự.
    constant_proba = float(y_train[0])
    ensemble_proba = np.full(len(kq_pred), constant_proba)
else:
    rf = RandomForestClassifier(
        n_estimators=400,
        max_depth=4,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train_s, y_train)

    if USE_XGB:
        neg = max(int((y_train == 0).sum()), 1)
        pos = max(int((y_train == 1).sum()), 1)
        boost = xgb.XGBClassifier(
            n_estimators=350,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            scale_pos_weight=neg / pos,
            random_state=42,
            verbosity=0,
        )
    else:
        boost = GradientBoostingClassifier(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.04,
            random_state=42,
        )

    boost.fit(X_train_s, y_train)
    proba_rf = rf.predict_proba(X_pred_s)[:, 1]
    proba_boost = boost.predict_proba(X_pred_s)[:, 1]
    ensemble_proba = (proba_rf + proba_boost) / 2


# ════════════════════════════════════════════════════════════════
# 5. TỔNG HỢP KẾT QUẢ BACKTEST
# ════════════════════════════════════════════════════════════════
def make_trade_row(stt: int, dataset_name: str, row: pd.Series, signal: str, confidence: str) -> dict:
    c_idx = int(row["candle_idx"])
    result = simulate_trade(c_idx, signal, use_trailing=True)

    sl_to_show = result["final_sl"] if SHOW_TRAILED_SL_IN_SL_COLUMN else result["initial_sl"]

    return {
        "STT": stt,
        "Tập dữ liệu": dataset_name,
        "Thời gian": str(row["time"])[:19],
        "Entry": round(float(result["entry"]), 2),
        "Tín hiệu": normalize_direction(signal),
        "Độ tin cậy": confidence,
        "SL": sl_to_show,
        "TP": result["tp"],
        "Kết quả": result["outcome"],
        "PnL": result["pnl"],
    }


all_trades_summary = []

# Tập train: giữ tín hiệu gốc từ scanner.
for idx, row in kq_train.iterrows():
    signal = normalize_direction(row["direction"])
    all_trades_summary.append(
        make_trade_row(
            stt=idx + 1,
            dataset_name="TRAIN (Gốc)",
            row=row,
            signal=signal,
            confidence="100%",
        )
    )

# Tập predict: AI quyết định giữ hoặc đảo chiều tín hiệu scanner.
for k, row in kq_pred.iterrows():
    prob_win = float(ensemble_proba[k])
    scanner_signal = normalize_direction(row["direction"])

    if prob_win >= AI_PROBA_THRESHOLD:
        final_signal = scanner_signal
        confidence = prob_win
    else:
        final_signal = "SHORT" if scanner_signal == "LONG" else "LONG"
        confidence = 1 - prob_win

    all_trades_summary.append(
        make_trade_row(
            stt=split + k + 1,
            dataset_name="PREDICT (AI)",
            row=row,
            signal=final_signal,
            confidence=f"{confidence * 100:.1f}%",
        )
    )


df_table = pd.DataFrame(all_trades_summary)
df_table.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# Metrics riêng cho AI
ai_df = df_table[df_table["Tập dữ liệu"] == "PREDICT (AI)"].copy()
ai_closed = ai_df[ai_df["Kết quả"] != "OPEN"].copy()
ai_wins = int((ai_closed["PnL"] > 0).sum())
ai_finished_trades = len(ai_closed)
ai_total_pnl = float(ai_closed["PnL"].sum()) if ai_finished_trades > 0 else 0.0
ai_win_rate = (ai_wins / ai_finished_trades * 100) if ai_finished_trades > 0 else 0.0


# ════════════════════════════════════════════════════════════════
# 6. DỰNG BẢNG BÁO CÁO: GIỮ NGUYÊN HIỂN THỊ GỐC NHƯ ẢNH
# ════════════════════════════════════════════════════════════════
print("\n🟢 Đang tạo bảng thống kê báo cáo siêu nhẹ...")

outcome_colors = []
pnl_colors = []
for _, r in df_table.iterrows():
    outcome = r["Kết quả"]
    pnl = float(r["PnL"])

    if outcome == "WIN":
        outcome_colors.append("#00FF88")
    elif outcome == "LOSS":
        outcome_colors.append("#FF4466")
    elif outcome == "BE":
        outcome_colors.append("#FBBF24")
    else:
        outcome_colors.append("#FFFFFF")

    if pnl > 0:
        pnl_colors.append("#00FF88")
    elif pnl < 0:
        pnl_colors.append("#FF4466")
    elif outcome == "BE":
        pnl_colors.append("#FBBF24")
    else:
        pnl_colors.append("#FFFFFF")

# Đảm bảo đúng thứ tự cột giống ảnh gốc.
display_columns = ["STT", "Tập dữ liệu", "Thời gian", "Entry", "Tín hiệu", "Độ tin cậy", "SL", "TP", "Kết quả", "PnL"]
df_display = df_table[display_columns]

fig = go.Figure(data=[
    go.Table(
        header=dict(
            values=[f"<b>{col}</b>" for col in df_display.columns],
            fill_color="#1F2937",
            align="center",
            font=dict(color="white", size=13),
            height=35,
        ),
        cells=dict(
            values=[df_display[col].tolist() for col in df_display.columns],
            fill_color="#111827",
            align="center",
            font=dict(
                color=[
                    ["#FFFFFF"] * len(df_display),   # STT
                    ["#E5E7EB"] * len(df_display),   # Tập dữ liệu
                    ["#9CA3AF"] * len(df_display),   # Thời gian
                    ["#FFFFFF"] * len(df_display),   # Entry
                    ["#38BDF8"] * len(df_display),   # Tín hiệu
                    ["#FBBF24"] * len(df_display),   # Độ tin cậy
                    ["#EF4444"] * len(df_display),   # SL
                    ["#34D399"] * len(df_display),   # TP
                    outcome_colors,                  # Kết quả
                    pnl_colors,                      # PnL
                ],
                size=12,
            ),
            height=28,
        ),
    )
])

dashboard_title = (
    "<b>SVD CONFLUENCE QUANT TRADING REPORT</b><br>"
    f"<span style='font-size:14px; color:#38BDF8;'>Tổng số lệnh: {len(kq)}</span> | "
    f"<span style='font-size:14px; color:#00FF88;'>[AI] Tỷ lệ thắng (WinRate): {ai_win_rate:.2f}%</span> | "
    f"<span style='font-size:14px; color:#FBBF24;'>[AI] Tổng điểm thắng ròng (Net PnL): {ai_total_pnl:.1f} điểm</span>"
)

fig.update_layout(
    title=dashboard_title,
    template="plotly_dark",
    margin=dict(l=20, r=20, t=85, b=20),
    height=850,
)

fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn", full_html=True)
fig.show()

print("\n📊 KẾT QUẢ TẬP AI (PREDICT):")
print(f"   ➤ Số lệnh đã đóng: {ai_finished_trades}")
print(f"   ➤ Tỉ lệ thắng: {ai_win_rate:.2f}%")
print(f"   ➤ Net Profit: {ai_total_pnl:.1f} điểm")
print("\n🧲 TRAILING SL:")
print(f"   ➤ Bật kéo SL: {USE_TRAILING_SL}")
print(f"   ➤ Hòa vốn khi lời: {BREAKEVEN_TRIGGER_POINTS} điểm")
print(f"   ➤ Trailing bắt đầu khi lời: {TRAIL_START_POINTS} điểm")
print(f"   ➤ Khoảng cách trailing: {TRAIL_DISTANCE_POINTS} điểm")
print(f"\n✅ Đã lưu: {OUTPUT_CSV}")
print(f"✅ Đã lưu: {OUTPUT_HTML}")
